# F2 · KNN, logistic and random forest on four shapes of data

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 60 minutes · **extended path**

## Objective

Compare three models (scaled KNN with `k=15`, logistic regression, random forest of 100 trees) on **four datasets of different shapes**: nonlinear boundary in 2D, linear boundary in 40 dimensions, nonlinear boundary with 30 noise columns, and a
large dataset. Measure accuracy, training time and prediction time, and summarize when each model is the right choice.

## Dataset

`datasets()` (given): dictionary name → `(X, y)` with `moons 2D` (3000 points, `noise=0.3`), `linear 40D` (3000 points, 40 features, linear boundary), `moons + 30 noise` (the moons with 30 noise columns), `large` (40,000 points, 20 features). 70/30 split.

## Educational purpose

Build the habit of **choosing the model by looking at the shape of the data** and not at the fashionable name: every model has a case where it wins and one where it loses, and the cost (training and prediction) is part of the choice.

## Task

1. For each of the four datasets and each of the three models (`knn`, `logistic`, `forest`) measure: `acc` (test accuracy), `fit_time` (seconds for `fit`) and `pred_time` (seconds for `predict` on the test set). Save `table` with index `(dataset, model)` and the three columns.
2. Save `best` (dictionary dataset → model with maximum accuracy) and `worst` (dictionary dataset → model with minimum accuracy).
3. Save `knn_fastest_fit` (True if on the `large` dataset the KNN's `fit_time` is the smallest of the three) and `pred_ratio` (on the `large` dataset: `pred_time` of the KNN divided by `pred_time` of the logistic).
4. Answer with a string chosen among `'knn'`, `'logistic'`, `'forest'`: `for_low_latency` (the model best suited to serving thousands of predictions per second on a large dataset) and `for_noise` (the most robust to many useless features, among the three).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
from sklearn.datasets import make_classification, make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def datasets():
    rng = np.random.default_rng(0)
    moons, y_moons = make_moons(3000, noise=0.3, random_state=0)
    Xl = rng.normal(size=(3000, 40))
    w = np.r_[rng.normal(size=10), np.zeros(30)]                     # only 10 features out of 40 matter, linearly
    yl = rng.binomial(1, 1 / (1 + np.exp(-3 * Xl @ w / np.linalg.norm(w))))
    large = make_classification(40000, n_features=20, n_informative=8, n_redundant=0, class_sep=1.0, random_state=0)
    return {"moons 2D": (moons, y_moons), "linear 40D": (Xl, yl), "moons + 30 noise": (np.c_[moons, rng.normal(size=(3000, 30))], y_moons), "large": large}


models = {
    "knn": lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
    "logistic": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "forest": lambda: RandomForestClassifier(100, random_state=0, n_jobs=-1),
}

## Your work

In [ ]:
rows = {}
for name, (X, y) in datasets().items():
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
    for model_name, make in models.items():
        ...
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
best = {}
worst = {}
knn_fastest_fit = None
pred_ratio = None
for_low_latency = None
for_noise = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='For each model: `m = make(); t0 = time.time(); m.fit(X_tr, y_tr); fit_time = time.time() - t0; t0 = time.time(); acc = m.score(X_te, y_te); pred_time = time.time() - t0`. Row: `rows[(name, model_name)] = {"acc": acc, "fit_time": fit_time, "pred_time": pred_time}`. Best: `table["acc"].unstack().idxmax(axis=1)` (with `unstack`, the indices become rows/columns).'):
    check(table.shape == (12, 3) and list(table.columns) == ["acc", "fit_time", "pred_time"], "12 rows (4 datasets x 3 models) and three columns", "acc, fit_time, pred_time")
    check(best["linear 40D"] == "logistic" and worst["linear 40D"] == "knn", "on the linear boundary in 40 dimensions the logistic wins and the KNN loses", "compare the accuracies")
    check(worst["moons + 30 noise"] == "knn" and best["moons + 30 noise"] == "forest", "with 30 noise columns the forest wins and the KNN is last", "compare the accuracies")
    check(table.loc[("moons 2D", "logistic"), "acc"] < table.loc[("moons 2D", "knn"), "acc"] - 0.03 and worst["moons 2D"] == "logistic", "on the 2D moons the logistic (linear boundary) loses to the KNN by over 3 points", "the boundary is not linear")
    check(knn_fastest_fit is True and pred_ratio > 20, "on the large dataset the KNN has the fastest fit and prediction over 20 times slower than the logistic", "KNN predict computes the distances from the whole training set")
    check(for_low_latency == "logistic" and for_noise == "forest", "low latency: logistic; robustness to noise: forest", "summary of the comparison")
done()

---
The commented solution is in `solutions/F_compare/F2_knn_logistic_forest_on_four_data_shapes.ipynb`. Open it after you pass the check, or after you have really tried.